# Mathematical Foundations of DBSCAN> **Difficulty**: Intermediate  > **Estimated Time**: 45-60 minutes  > **Prerequisites**: Basic understanding of clustering, Euclidean distance, set theory notation## Learning ObjectivesBy the end of this notebook, you will be able to:- Understand formal mathematical definitions of DBSCAN concepts- Visualize epsilon-neighborhoods and their properties- Identify core points using the formal definition- Apply mathematical concepts to real datasets- Solve problems involving DBSCAN mathematical foundations## Paper ReferencesThis notebook covers concepts from:- Section 4.1: Density-Based Notions of Clusters (p. 227)- Core definitions and mathematical formulations## Table of Contents1. [Setup and Imports](#setup)2. [Epsilon-Neighborhood Definition](#epsilon-neighborhood)3. [Core Point Condition](#core-point)4. [Distance Metrics](#distance-metrics)5. [Interactive Exploration](#interactive)6. [Exercises](#exercises)7. [Summary](#summary)

## 1. Setup and Imports <a id='setup'></a>First, let's import the necessary libraries and modules.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport syssys.path.append('..')from src.dbscan_from_scratch import DBSCANfrom src.data_loader import load_sample_datafrom src.visualization import DBSCANVisualizer# Set random seed for reproducibilitynp.random.seed(42)# Create visualizerviz = DBSCANVisualizer()print("Setup complete!")

## 2. Epsilon-Neighborhood Definition <a id='epsilon-neighborhood'></a>### Formal Definition [Paper §4.1, p. 227]The **ε-neighborhood** of a point p, denoted N_ε(p), is defined as:```N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}```Where:- **D** is the database (dataset) of points- **dist(p, q)** is the distance function between points p and q- **ε** (epsilon) is the radius parameter**Intuition**: The ε-neighborhood is the set of all points within distance ε from point p, including p itself.**Example**: If ε = 1.0 and we use Euclidean distance, N_ε(p) contains all points within a circle of radius 1.0 centered at p.### VisualizationLet's visualize the ε-neighborhood concept with a simple dataset.

In [ ]:
# Generate a simple datasetX = load_sample_data('moons', n_samples=100)# Parameterseps = 0.3point_idx = 10  # Point to visualize neighborhood for# Visualize epsilon-neighborhoodviz.plot_epsilon_neighborhood(X, point_idx, eps)plt.show()# Calculate the neighborhood manuallydistances = np.sqrt(np.sum((X - X[point_idx])**2, axis=1))neighbors = np.where(distances <= eps)[0]print(f"Point {point_idx} coordinates: {X[point_idx]}")print(f"Epsilon (ε): {eps}")print(f"|N_ε(p)|: {len(neighbors)} points")print(f"Neighbor indices: {neighbors}")

### Mathematical Properties of ε-Neighborhood1. **Self-inclusion**: p ∈ N_ε(p) (a point is always in its own neighborhood)2. **Symmetry**: If q ∈ N_ε(p), then p ∈ N_ε(q) (for symmetric distance metrics)3. **Cardinality**: |N_ε(p)| ≥ 1 (at minimum, contains the point itself)4. **Monotonicity**: If ε₁ < ε₂, then N_ε₁(p) ⊆ N_ε₂(p) (larger radius includes more points)Let's verify these properties:

In [ ]:
# Property 1: Self-inclusionpoint_idx = 5distances = np.sqrt(np.sum((X - X[point_idx])**2, axis=1))neighbors = np.where(distances <= eps)[0]print(f"Property 1 - Self-inclusion: {point_idx in neighbors}")# Property 2: Symmetry (for Euclidean distance)point_a, point_b = 5, 10dist_ab = np.sqrt(np.sum((X[point_a] - X[point_b])**2))dist_ba = np.sqrt(np.sum((X[point_b] - X[point_a])**2))print(f"Property 2 - Symmetry: dist(a,b) = {dist_ab:.4f}, dist(b,a) = {dist_ba:.4f}")# Property 3: Cardinalityprint(f"Property 3 - Cardinality: |N_ε(p)| = {len(neighbors)} >= 1")# Property 4: Monotonicityeps1, eps2 = 0.2, 0.4neighbors1 = np.where(np.sqrt(np.sum((X - X[point_idx])**2, axis=1)) <= eps1)[0]neighbors2 = np.where(np.sqrt(np.sum((X - X[point_idx])**2, axis=1)) <= eps2)[0]print(f"Property 4 - Monotonicity: |N_{eps1}(p)| = {len(neighbors1)}, |N_{eps2}(p)| = {len(neighbors2)}")print(f"  N_ε₁(p) ⊆ N_ε₂(p): {set(neighbors1).issubset(set(neighbors2))}")

## 3. Core Point Condition <a id='core-point'></a>### Formal Definition [Paper §4.1, p. 227]A point p is a **core point** if its ε-neighborhood contains at least MinPts points:```|N_ε(p)| ≥ MinPts```Where:- **|N_ε(p)|** denotes the cardinality (size) of the ε-neighborhood- **MinPts** is the minimum number of points required to form a dense region**Intuition**: Core points are in dense regions of the dataset. They have enough neighbors to be considered part of a cluster's "core."### Point Type ClassificationDBSCAN classifies points into three types:1. **Core Point**: |N_ε(p)| ≥ MinPts2. **Border Point**: Not a core point, but within ε of a core point3. **Noise Point**: Neither core nor border (outlier)Let's visualize these point types:

In [ ]:
# Run DBSCAN to classify pointseps = 0.3min_pts = 5dbscan = DBSCAN(eps=eps, min_pts=min_pts)labels = dbscan.fit_predict(X)core_indices = dbscan.get_core_points()# Visualize point typesviz.plot_point_types(X, labels, core_indices, eps)plt.show()# Count point typescore_mask = np.zeros(len(X), dtype=bool)core_mask[core_indices] = Truenoise_mask = labels == -1border_mask = ~core_mask & ~noise_maskprint(f"\nPoint Type Distribution:")print(f"  Core points: {np.sum(core_mask)} ({100*np.sum(core_mask)/len(X):.1f}%)")print(f"  Border points: {np.sum(border_mask)} ({100*np.sum(border_mask)/len(X):.1f}%)")print(f"  Noise points: {np.sum(noise_mask)} ({100*np.sum(noise_mask)/len(X):.1f}%)")

### Core Point Condition AnalysisLet's examine how the core point condition works in detail:

In [ ]:
# Analyze a specific pointpoint_idx = core_indices[0] if len(core_indices) > 0 else 0# Calculate neighborhooddistances = np.sqrt(np.sum((X - X[point_idx])**2, axis=1))neighbors = np.where(distances <= eps)[0]print(f"Analyzing Point {point_idx}:")print(f"  Coordinates: {X[point_idx]}")print(f"  |N_ε(p)|: {len(neighbors)}")print(f"  MinPts: {min_pts}")print(f"  Is core point: {len(neighbors) >= min_pts}")print(f"  Point type: {dbscan.get_point_type(point_idx).name}")# Visualize this specific point's neighborhoodviz.plot_epsilon_neighborhood(X, point_idx, eps, labels)plt.title(f"Point {point_idx}: {dbscan.get_point_type(point_idx).name} Point")plt.show()

## 4. Distance Metrics <a id='distance-metrics'></a>### Distance Function DefinitionThe distance function dist(p, q) can use various metrics. The paper uses Euclidean distance by default [Paper §4.1, p. 227].### Euclidean Distance (L2 norm)```d(p, q) = √(Σᵢ(pᵢ - qᵢ)²)```For 2D data:```d(p, q) = √((x₁ - x₂)² + (y₁ - y₂)²)```**Example**: Distance between (0, 0) and (3, 4) is √(9 + 16) = 5### Alternative Metrics**Manhattan Distance** (L1 norm):```d(p, q) = Σᵢ|pᵢ - qᵢ|```**Chebyshev Distance** (L∞ norm):```d(p, q) = maxᵢ|pᵢ - qᵢ|```Let's compare these distance metrics:

In [ ]:
# Define two pointsp1 = np.array([0.0, 0.0])p2 = np.array([3.0, 4.0])# Calculate distances using different metricsdbscan_euclidean = DBSCAN(eps=0.5, min_pts=5, metric='euclidean')dbscan_manhattan = DBSCAN(eps=0.5, min_pts=5, metric='manhattan')dbscan_chebyshev = DBSCAN(eps=0.5, min_pts=5, metric='chebyshev')dist_euclidean = dbscan_euclidean._compute_distance(p1, p2)dist_manhattan = dbscan_manhattan._compute_distance(p1, p2)dist_chebyshev = dbscan_chebyshev._compute_distance(p1, p2)print(f"Distance between {p1} and {p2}:")print(f"  Euclidean: {dist_euclidean:.4f}")print(f"  Manhattan: {dist_manhattan:.4f}")print(f"  Chebyshev: {dist_chebyshev:.4f}")# Visualize the differencefig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, metric, dist in zip(axes,                              ['euclidean', 'manhattan', 'chebyshev'],                             [dist_euclidean, dist_manhattan, dist_chebyshev]):    ax.scatter([p1[0]], [p1[1]], c='red', s=200, marker='*', label='Point 1', zorder=5)    ax.scatter([p2[0]], [p2[1]], c='blue', s=200, marker='*', label='Point 2', zorder=5)    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k--', alpha=0.5)    ax.set_xlim(-1, 5)    ax.set_ylim(-1, 5)    ax.set_aspect('equal')    ax.grid(True, alpha=0.3)    ax.set_title(f'{metric.capitalize()}\nd = {dist:.2f}')    ax.legend()plt.tight_layout()plt.show()

### Impact of Distance Metric on ClusteringDifferent distance metrics can produce different clustering results:

In [ ]:
# Compare clustering with different metricsX_test = load_sample_data('moons', n_samples=150)eps = 0.3min_pts = 5fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, metric in zip(axes, ['euclidean', 'manhattan', 'chebyshev']):    dbscan = DBSCAN(eps=eps, min_pts=min_pts, metric=metric)    labels = dbscan.fit_predict(X_test)        # Plot    unique_labels = set(labels)    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))        for label, color in zip(unique_labels, colors):        if label == -1:            ax.scatter(X_test[labels == label, 0], X_test[labels == label, 1],                      c='black', marker='x', s=50, alpha=0.5, label='Noise')        else:            ax.scatter(X_test[labels == label, 0], X_test[labels == label, 1],                      c=[color], s=50, alpha=0.7, label=f'Cluster {label}')        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)    n_noise = list(labels).count(-1)    ax.set_title(f'{metric.capitalize()}\n{n_clusters} clusters, {n_noise} noise')    ax.set_xlabel('Feature 1')    ax.set_ylabel('Feature 2')    ax.legend(loc='best')plt.tight_layout()plt.show()

## 5. Interactive Exploration <a id='interactive'></a>### Exploring Parameter EffectsLet's explore how changing ε and MinPts affects the mathematical properties.

In [ ]:
# Test different epsilon valuesX_test = load_sample_data('moons', n_samples=100)point_idx = 20eps_values = [0.2, 0.3, 0.5, 0.8]fig, axes = plt.subplots(2, 2, figsize=(12, 10))axes = axes.flatten()for ax, eps in zip(axes, eps_values):    # Calculate neighborhood    distances = np.sqrt(np.sum((X_test - X_test[point_idx])**2, axis=1))    neighbors = np.where(distances <= eps)[0]        # Plot    ax.scatter(X_test[:, 0], X_test[:, 1], c='lightgray', s=30, alpha=0.5)    ax.scatter(X_test[point_idx, 0], X_test[point_idx, 1],               c='red', marker='*', s=300, edgecolors='black', linewidths=2, zorder=5)    if len(neighbors) > 1:        neighbors_no_self = neighbors[neighbors != point_idx]        ax.scatter(X_test[neighbors_no_self, 0], X_test[neighbors_no_self, 1],                  facecolors='none', edgecolors='orange', s=150, linewidths=2, zorder=4)        # Draw circle    circle = plt.Circle((X_test[point_idx, 0], X_test[point_idx, 1]), eps,                       color='gray', fill=False, linestyle='--', linewidth=2, alpha=0.5)    ax.add_patch(circle)        ax.set_title(f'ε = {eps}\n|N_ε(p)| = {len(neighbors)}')    ax.set_xlabel('Feature 1')    ax.set_ylabel('Feature 2')    ax.set_aspect('equal')    ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Observation: As ε increases, |N_ε(p)| increases (monotonicity property)")

### MinPts Threshold EffectNow let's see how MinPts affects core point classification:

In [ ]:
# Test different MinPts valueseps = 0.3minpts_values = [3, 5, 7, 10]fig, axes = plt.subplots(2, 2, figsize=(12, 10))axes = axes.flatten()for ax, min_pts in zip(axes, minpts_values):    dbscan = DBSCAN(eps=eps, min_pts=min_pts)    labels = dbscan.fit_predict(X_test)    core_indices = dbscan.get_core_points()        # Count point types    core_mask = np.zeros(len(X_test), dtype=bool)    core_mask[core_indices] = True    noise_mask = labels == -1    border_mask = ~core_mask & ~noise_mask        # Plot    if np.any(noise_mask):        ax.scatter(X_test[noise_mask, 0], X_test[noise_mask, 1],                  c='black', marker='x', s=50, alpha=0.6, label='Noise')    if np.any(border_mask):        ax.scatter(X_test[border_mask, 0], X_test[border_mask, 1],                  c='green', marker='o', s=60, alpha=0.7, label='Border')    if np.any(core_mask):        ax.scatter(X_test[core_mask, 0], X_test[core_mask, 1],                  c='blue', marker='o', s=100, alpha=0.8,                   edgecolors='black', linewidths=0.5, label='Core')        ax.set_title(f'MinPts = {min_pts}\nCore: {np.sum(core_mask)}, Border: {np.sum(border_mask)}, Noise: {np.sum(noise_mask)}')    ax.set_xlabel('Feature 1')    ax.set_ylabel('Feature 2')    ax.legend(loc='best')    ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Observation: As MinPts increases, fewer points qualify as core points")

## 6. Exercises <a id='exercises'></a>Test your understanding of DBSCAN mathematical foundations with these exercises.### Exercise 1: Epsilon-Neighborhood Calculation (Beginner)Given the following 2D points and ε = 2.0:- Point A: (0, 0)- Point B: (1, 1)- Point C: (3, 0)- Point D: (0, 3)**Question**: Calculate |N_ε(A)| (the size of A's epsilon-neighborhood).<details><summary>Click to reveal solution</summary>**Solution**:

In [ ]:
# Exercise 1 Solutionpoints = np.array([[0, 0], [1, 1], [3, 0], [0, 3]])point_names = ['A', 'B', 'C', 'D']eps = 2.0point_A_idx = 0# Calculate distances from A to all pointsdistances = np.sqrt(np.sum((points - points[point_A_idx])**2, axis=1))print("Exercise 1 Solution:")print(f"\nDistances from A to all points:")for i, (name, dist) in enumerate(zip(point_names, distances)):    print(f"  dist(A, {name}) = {dist:.2f} {'<= ε' if dist <= eps else '> ε'}")neighbors = np.where(distances <= eps)[0]print(f"\nN_ε(A) = {{{', '.join([point_names[i] for i in neighbors])}}}")print(f"|N_ε(A)| = {len(neighbors)}")# Visualizeplt.figure(figsize=(8, 8))plt.scatter(points[:, 0], points[:, 1], c='blue', s=200, zorder=5)for i, name in enumerate(point_names):    plt.annotate(name, (points[i, 0], points[i, 1]),                 xytext=(5, 5), textcoords='offset points', fontsize=14, fontweight='bold')# Draw epsilon circle around Acircle = plt.Circle((points[point_A_idx, 0], points[point_A_idx, 1]), eps,                   color='red', fill=False, linestyle='--', linewidth=2, alpha=0.5)plt.gca().add_patch(circle)plt.scatter([points[point_A_idx, 0]], [points[point_A_idx, 1]],            c='red', s=300, marker='*', zorder=6, label='Point A')plt.xlim(-1, 4)plt.ylim(-1, 4)plt.xlabel('X')plt.ylabel('Y')plt.title(f'ε-neighborhood of A (ε = {eps})')plt.legend()plt.grid(True, alpha=0.3)plt.axis('equal')plt.show()

</details>### Exercise 2: Core Point Identification (Intermediate)Using the same points from Exercise 1 with ε = 2.0 and MinPts = 3:**Question**: Which points are core points?<details><summary>Click to reveal solution</summary>**Solution**:

In [ ]:
# Exercise 2 Solutioneps = 2.0min_pts = 3print("Exercise 2 Solution:")print(f"\nParameters: ε = {eps}, MinPts = {min_pts}")print(f"\nAnalyzing each point:")for i, name in enumerate(point_names):    distances = np.sqrt(np.sum((points - points[i])**2, axis=1))    neighbors = np.where(distances <= eps)[0]    is_core = len(neighbors) >= min_pts        print(f"\nPoint {name}:")    print(f"  Neighbors: {{{', '.join([point_names[j] for j in neighbors])}}}")    print(f"  |N_ε({name})| = {len(neighbors)}")    print(f"  Is core point: {is_core} ({'|N_ε(p)| >= MinPts' if is_core else '|N_ε(p)| < MinPts'})")

</details>### Exercise 3: Distance Metric Comparison (Advanced)Given points P = (1, 2) and Q = (4, 6):**Question**: Calculate the distance using Euclidean, Manhattan, and Chebyshev metrics. Which metric gives the smallest distance?<details><summary>Click to reveal solution</summary>**Solution**:

In [ ]:
# Exercise 3 SolutionP = np.array([1.0, 2.0])Q = np.array([4.0, 6.0])# Calculate distancesdbscan_euc = DBSCAN(metric='euclidean')dbscan_man = DBSCAN(metric='manhattan')dbscan_cheb = DBSCAN(metric='chebyshev')dist_euclidean = dbscan_euc._compute_distance(P, Q)dist_manhattan = dbscan_man._compute_distance(P, Q)dist_chebyshev = dbscan_cheb._compute_distance(P, Q)print("Exercise 3 Solution:")print(f"\nPoint P: {P}")print(f"Point Q: {Q}")print(f"\nDistances:")print(f"  Euclidean: √((4-1)² + (6-2)²) = √(9 + 16) = √25 = {dist_euclidean:.2f}")print(f"  Manhattan: |4-1| + |6-2| = 3 + 4 = {dist_manhattan:.2f}")print(f"  Chebyshev: max(|4-1|, |6-2|) = max(3, 4) = {dist_chebyshev:.2f}")min_dist = min(dist_euclidean, dist_manhattan, dist_chebyshev)min_metric = ['Euclidean', 'Manhattan', 'Chebyshev'][[dist_euclidean, dist_manhattan, dist_chebyshev].index(min_dist)]print(f"\nSmallest distance: {min_metric} ({min_dist:.2f})")

</details>### Exercise 4: Mathematical Properties (Advanced)**Question**: Prove that if ε₁ < ε₂, then N_ε₁(p) ⊆ N_ε₂(p) (monotonicity property).<details><summary>Click to reveal solution</summary>**Solution**:**Proof**:Let q ∈ N_ε₁(p). By definition of ε-neighborhood:- q ∈ N_ε₁(p) ⟹ dist(p, q) ≤ ε₁Since ε₁ < ε₂:- dist(p, q) ≤ ε₁ < ε₂- Therefore, dist(p, q) ≤ ε₂- By definition, q ∈ N_ε₂(p)Since this holds for any q ∈ N_ε₁(p), we have N_ε₁(p) ⊆ N_ε₂(p). ∎**Verification with code**:

In [ ]:
# Exercise 4 VerificationX_verify = load_sample_data('moons', n_samples=50)point_idx = 10eps1, eps2 = 0.3, 0.5# Calculate neighborhoodsdistances = np.sqrt(np.sum((X_verify - X_verify[point_idx])**2, axis=1))N_eps1 = set(np.where(distances <= eps1)[0])N_eps2 = set(np.where(distances <= eps2)[0])print("Exercise 4 Verification:")print(f"\nε₁ = {eps1}, ε₂ = {eps2}")print(f"|N_ε₁(p)| = {len(N_eps1)}")print(f"|N_ε₂(p)| = {len(N_eps2)}")print(f"\nN_ε₁(p) ⊆ N_ε₂(p): {N_eps1.issubset(N_eps2)}")print(f"\nThis verifies the monotonicity property!")

</details>

## 7. Summary <a id='summary'></a>### Key Takeaways1. **ε-Neighborhood** [Paper §4.1, p. 227]:   - Definition: N_ε(p) = {q ∈ D | dist(p, q) ≤ ε}   - Contains all points within distance ε from p   - Properties: self-inclusion, symmetry, monotonicity2. **Core Point Condition** [Paper §4.1, p. 227]:   - Definition: |N_ε(p)| ≥ MinPts   - Core points form the "skeleton" of clusters   - Classification: Core, Border, or Noise3. **Distance Metrics**:   - Euclidean: d(p,q) = √(Σ(pᵢ-qᵢ)²)   - Manhattan: d(p,q) = Σ|pᵢ-qᵢ|   - Chebyshev: d(p,q) = max|pᵢ-qᵢ|   - Different metrics can produce different clustering results4. **Mathematical Properties**:   - ε-neighborhoods are well-defined mathematical sets   - Core point condition provides a formal density threshold   - Properties can be proven mathematically and verified empirically### Next StepsAfter completing this notebook, you should:1. **For density concepts**: Proceed to `04_density_concepts.ipynb` to learn about density-reachability and density-connectivity2. **For algorithm details**: See `05_algorithm_walkthrough.ipynb` for step-by-step algorithm execution3. **For theory**: Read `docs/02_density_concepts.md` for formal definitions and proofs4. **For implementation**: Study `src/dbscan_from_scratch.py` to see how these concepts are implemented### References**Primary Reference**:- Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96)* (Vol. 96, No. 34, pp. 226-231).**Specific Sections**:- §4.1 (p. 227): Density-based notions of clusters - formal definitions**Paper Location**: `../1996-DBSCAN-KDD.pdf` in repository root